Baseline Linear Regression Model – Full Feature Set

structured_traffic.csv

This model serves as an initial baseline linear regression model to predict traffic volume using a comprehensive set of available features from the structured traffic dataset. The predictors include temporal characteristics such as hour, day of week, month, and weekend status, along with roadway-specific information including roadway name, direction, and segment identifier.

Categorical variables were converted into indexed numerical representations using Spark ML’s StringIndexer, and all features were combined into a single feature vector using VectorAssembler. This baseline model provides a benchmark for understanding how much variation in traffic volume can be explained by a linear combination of both temporal and spatial predictors.

In [0]:
df_raw = spark.read.csv(
    "/Volumes/workspace/default/datafiles/structured_traffic.csv",
    header=True,
    inferSchema=True
)

df_raw.show(5)

+----------+-------------+-----------+----------------+---------+----------+----+----------+-----------+-------+-----+----------+-------------------+--------------+
|segment_id|roadway__name|       from|              to|direction|      date|hour|hour_start|day_of_week|weekday|month|is_weekend|          timestamp|traffic_volume|
+----------+-------------+-----------+----------------+---------+----------+----+----------+-----------+-------+-----+----------+-------------------+--------------+
|     15540| BEACH STREET|UNION PLACE|VAN DUZER STREET|       NB|2012-01-09|   1|   1:00 AM|     Monday|      0|    1|     false|2012-01-09 01:00:00|          20.0|
|     15540| BEACH STREET|UNION PLACE|VAN DUZER STREET|       NB|2012-01-10|   1|   1:00 AM|    Tuesday|      1|    1|     false|2012-01-10 01:00:00|          21.0|
|     15540| BEACH STREET|UNION PLACE|VAN DUZER STREET|       NB|2012-01-11|   1|   1:00 AM|  Wednesday|      2|    1|     false|2012-01-11 01:00:00|          27.0|
|     1554

In [0]:
df_raw.printSchema()

root
 |-- segment_id: integer (nullable = true)
 |-- roadway__name: string (nullable = true)
 |-- from: string (nullable = true)
 |-- to: string (nullable = true)
 |-- direction: string (nullable = true)
 |-- date: date (nullable = true)
 |-- hour: integer (nullable = true)
 |-- hour_start: string (nullable = true)
 |-- day_of_week: string (nullable = true)
 |-- weekday: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- is_weekend: boolean (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- traffic_volume: double (nullable = true)



In [0]:
for c in df_raw.columns:
    print(repr(c))

'segment_id'
'roadway__name'
'from'
'to'
'direction'
'date'
'hour'
'hour_start'
'day_of_week'
'weekday'
'month'
'is_weekend'
'timestamp'
'traffic_volume'


In [0]:
road_col = [c for c in df_raw.columns if "roadway" in c.lower()][0]
print("Detected roadway column:", repr(road_col))

Detected roadway column: 'roadway__name'


In [0]:
from pyspark.sql.functions import col

df = df_raw.select(
    col("traffic_volume"),
    col("hour"),
    col("direction"),
    col(road_col).alias("roadway_name"),
    col("day_of_week"),
    col("is_weekend"),
    col("month"),
    col("segment_id")
)

df.show(5)

+--------------+----+---------+------------+-----------+----------+-----+----------+
|traffic_volume|hour|direction|roadway_name|day_of_week|is_weekend|month|segment_id|
+--------------+----+---------+------------+-----------+----------+-----+----------+
|          20.0|   1|       NB|BEACH STREET|     Monday|     false|    1|     15540|
|          21.0|   1|       NB|BEACH STREET|    Tuesday|     false|    1|     15540|
|          27.0|   1|       NB|BEACH STREET|  Wednesday|     false|    1|     15540|
|          22.0|   1|       NB|BEACH STREET|   Thursday|     false|    1|     15540|
|          31.0|   1|       NB|BEACH STREET|     Friday|     false|    1|     15540|
+--------------+----+---------+------------+-----------+----------+-----+----------+
only showing top 5 rows


In [0]:
df = df.na.drop()

df.show(5)
print("Row count after dropping missing values:", df.count())

+--------------+----+---------+------------+-----------+----------+-----+----------+
|traffic_volume|hour|direction|roadway_name|day_of_week|is_weekend|month|segment_id|
+--------------+----+---------+------------+-----------+----------+-----+----------+
|          20.0|   1|       NB|BEACH STREET|     Monday|     false|    1|     15540|
|          21.0|   1|       NB|BEACH STREET|    Tuesday|     false|    1|     15540|
|          27.0|   1|       NB|BEACH STREET|  Wednesday|     false|    1|     15540|
|          22.0|   1|       NB|BEACH STREET|   Thursday|     false|    1|     15540|
|          31.0|   1|       NB|BEACH STREET|     Friday|     false|    1|     15540|
+--------------+----+---------+------------+-----------+----------+-----+----------+
only showing top 5 rows
Row count after dropping missing values: 1023064


In [0]:
df.printSchema()

root
 |-- traffic_volume: double (nullable = true)
 |-- hour: integer (nullable = true)
 |-- direction: string (nullable = true)
 |-- roadway_name: string (nullable = true)
 |-- day_of_week: string (nullable = true)
 |-- is_weekend: boolean (nullable = true)
 |-- month: integer (nullable = true)
 |-- segment_id: integer (nullable = true)



In [0]:
from pyspark.ml.feature import StringIndexer

indexer_direction = StringIndexer(
    inputCol="direction",
    outputCol="direction_idx",
    handleInvalid="keep"
)

indexer_roadway = StringIndexer(
    inputCol="roadway_name",
    outputCol="roadway_name_idx",
    handleInvalid="keep"
)

indexer_day = StringIndexer(
    inputCol="day_of_week",
    outputCol="day_of_week_idx",
    handleInvalid="keep"
)

In [0]:
df1 = indexer_direction.fit(df).transform(df)
df2 = indexer_roadway.fit(df1).transform(df1)
df3 = indexer_day.fit(df2).transform(df2)

df4 = df3

df4.select(
    "traffic_volume",
    "hour",
    "direction_idx",
    "roadway_name_idx",
    "day_of_week_idx",
    "is_weekend",
    "month",
    "segment_id"
).show(5)

+--------------+----+-------------+----------------+---------------+----------+-----+----------+
|traffic_volume|hour|direction_idx|roadway_name_idx|day_of_week_idx|is_weekend|month|segment_id|
+--------------+----+-------------+----------------+---------------+----------+-----+----------+
|          20.0|   1|          0.0|           473.0|            6.0|     false|    1|     15540|
|          21.0|   1|          0.0|           473.0|            5.0|     false|    1|     15540|
|          27.0|   1|          0.0|           473.0|            4.0|     false|    1|     15540|
|          22.0|   1|          0.0|           473.0|            3.0|     false|    1|     15540|
|          31.0|   1|          0.0|           473.0|            2.0|     false|    1|     15540|
+--------------+----+-------------+----------------+---------------+----------+-----+----------+
only showing top 5 rows


In [0]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=[
        "hour",
        "direction_idx",
        "roadway_name_idx",
        "day_of_week_idx",
        "is_weekend",
        "month",
        "segment_id"
    ],
    outputCol="features"
)

final_data = assembler.transform(df4).select("features", "traffic_volume")
final_data.show(5, truncate=False)

+-----------------------------------+--------------+
|features                           |traffic_volume|
+-----------------------------------+--------------+
|[1.0,0.0,473.0,6.0,0.0,1.0,15540.0]|20.0          |
|[1.0,0.0,473.0,5.0,0.0,1.0,15540.0]|21.0          |
|[1.0,0.0,473.0,4.0,0.0,1.0,15540.0]|27.0          |
|[1.0,0.0,473.0,3.0,0.0,1.0,15540.0]|22.0          |
|[1.0,0.0,473.0,2.0,0.0,1.0,15540.0]|31.0          |
+-----------------------------------+--------------+
only showing top 5 rows


In [0]:
train_data, test_data = final_data.randomSplit([0.7, 0.3], seed=42)

print("Training rows:", train_data.count())
print("Testing rows:", test_data.count())

Training rows: 716559
Testing rows: 306505


In [0]:
from pyspark.ml.regression import LinearRegression

lr = LinearRegression(
    featuresCol="features",
    labelCol="traffic_volume"
)

lr_model = lr.fit(train_data)

In [0]:
lr_pred = lr_model.transform(test_data)
lr_pred.select("traffic_volume", "prediction").show(10)

+--------------+------------------+
|traffic_volume|        prediction|
+--------------+------------------+
|         313.0|308.82582351596886|
|         309.0| 323.5081526339093|
|         270.0|323.89985467956285|
|         281.0|323.89985467956285|
|         386.0| 326.8652758871089|
|         785.0|326.87584793528026|
|         273.0| 326.8844281482889|
|         409.0| 341.9400731411501|
|         344.0|344.90626043914335|
|         216.0| 344.9125806853327|
+--------------+------------------+
only showing top 10 rows


In [0]:
from pyspark.ml.evaluation import RegressionEvaluator

rmse_eval = RegressionEvaluator(
    labelCol="traffic_volume",
    predictionCol="prediction",
    metricName="rmse"
)

r2_eval = RegressionEvaluator(
    labelCol="traffic_volume",
    predictionCol="prediction",
    metricName="r2"
)

mae_eval = RegressionEvaluator(
    labelCol="traffic_volume",
    predictionCol="prediction",
    metricName="mae"
)

lr_rmse = rmse_eval.evaluate(lr_pred)
lr_r2 = r2_eval.evaluate(lr_pred)
lr_mae = mae_eval.evaluate(lr_pred)

print("Baseline Linear Regression Results")
print("RMSE:", lr_rmse)
print("MAE:", lr_mae)
print("R2:", lr_r2)

Baseline Linear Regression Results
RMSE: 621.2951685971261
MAE: 356.8467416176029
R2: 0.07338054415909534


In [0]:
print("Intercept:", lr_model.intercept)

Intercept: 198.79326598018721


In [0]:
feature_names = [
    "hour",
    "direction_idx",
    "roadway_name_idx",
    "day_of_week_idx",
    "is_weekend",
    "month",
    "segment_id"
]

for name, coef in zip(feature_names, lr_model.coefficients):
    print(f"{name}: {coef}")

hour: 18.29885025166219
direction_idx: -16.20377250407391
roadway_name_idx: -0.19949816716510183
day_of_week_idx: -8.548035091019706
is_weekend: -76.70343272550996
month: 18.0402184615872
segment_id: 3.8304522360024706e-05
